# Incident Genome: Exploratory Data Analysis

**DSA210 Term Project** -- Alper Kilic

I scraped incident data from 14 cloud service status pages (GitHub, Cloudflare, Discord, etc.) to understand what makes some outages last minutes and others drag on for hours.

This notebook walks through the full EDA pipeline: cleaning, distributions, temporal patterns, correlations, hypothesis tests, and a data leakage analysis.

## 1. Setup

Imports, plot style, and loading the raw JSON data.

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, kruskal

plt.style.use("dark_background")
sns.set_palette("Set2")

DATA_DIR = Path("data")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

df = pd.read_json(DATA_DIR / "incidents.json")
print(f"Loaded {len(df)} incidents from {df['service'].nunique()} services")
print(f"Columns: {list(df.columns)}")
df.head(3)

## 2. Data Cleaning

Some incidents are still in progress or unresolved, so they don't have a meaningful duration. I'll filter those out, convert timestamps, and flag outliers using the IQR method.

In [ ]:
# missing values
print("=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

null_duration = df["duration_minutes"].isnull().sum()
print(f"\nNull duration_minutes: {null_duration}")
if null_duration > 0:
    print("Unresolved incidents:")
    print(df[df["duration_minutes"].isnull()][["service", "name", "status"]])

In [ ]:
# keep only resolved incidents
before = len(df)
df = df[df["status"] == "resolved"].copy()
after = len(df)
print(f"Filtered: {before} -> {after} rows (dropped {before - after} unresolved)")

df = df.dropna(subset=["duration_minutes"])
print(f"After dropping null durations: {len(df)} rows")

# remove broken records where resolved < created
df = df[df["duration_minutes"] >= 0]
print(f"After removing negative durations: {len(df)} rows")

# convert timestamps
df["created_at"] = pd.to_datetime(df["created_at"], format="mixed", utc=True)
df["resolved_at"] = pd.to_datetime(df["resolved_at"], format="mixed", utc=True)

In [ ]:
# outlier detection with IQR
q1 = df["duration_minutes"].quantile(0.25)
q3 = df["duration_minutes"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f"Q1: {q1:.1f}, Q3: {q3:.1f}, IQR: {iqr:.1f}")
print(f"Outlier bounds: [{lower:.1f}, {upper:.1f}]")

df["is_outlier"] = (df["duration_minutes"] < lower) | (df["duration_minutes"] > upper)
n_outliers = df["is_outlier"].sum()
print(f"\nOutliers: {n_outliers}")

if n_outliers > 0:
    print(df[df["is_outlier"]][["service", "name", "duration_minutes", "impact"]].to_string())

# keep outliers flagged but don't remove -- we'll need them for comparison later
print(f"\nWith outliers:    mean={df['duration_minutes'].mean():.1f} min, median={df['duration_minutes'].median():.1f} min")
df_no_outliers = df[~df["is_outlier"]]
print(f"Without outliers: mean={df_no_outliers['duration_minutes'].mean():.1f} min, median={df_no_outliers['duration_minutes'].median():.1f} min")

# save cleaned data
df.to_csv(DATA_DIR / "incidents_clean.csv", index=False)
print(f"\nSaved to incidents_clean.csv ({len(df)} rows)")

### 2a. First-hour feature and leakage notes

Instructor proposal feedback flagged **data leakage** in `num_updates`: this column counts every status-page update posted during the entire incident, including those written AFTER the issue was resolved. The count therefore grows with duration, which is exactly what we want to predict.

Fix: derive **`first_hour_updates`** — the number of status updates posted in the first 3600 s after the incident starts. This is the analog feature available at prediction time (t=0..1h). We also keep `num_updates` in the dataset for the leakage comparison in §10, but **downstream hypothesis tests and the correlation heatmap use the clean feature.**

Other leakage hygiene:

- **`impact`** on statuspage.io is the *highest* severity reached during the lifecycle, not the initial triage label. We flag this as a leaky label for the ML milestone — it will NOT be used as a feature, only as a stratification target.
- **`num_components`** reflects the final (cumulative) affected component set. Same caveat: usable as context, not as a t=0 feature.
- **66 incidents have duration = 0 min.** Manual spot-check shows these are auto-closed / status-page cleanup entries, not real outages; we keep them flagged so the downstream class-imbalance analysis can see them, but we rely on the IQR outlier mask to exclude them from the day-of-week averages.


In [ ]:
# build raw lookup keyed by (service, id) — id is unique, name can collide
raw_dir = DATA_DIR / "raw"
raw_lookup = {}
for fname in sorted(os.listdir(raw_dir)):
    if not fname.endswith("_raw.json"):
        continue
    service = fname.replace("_raw.json", "")
    with open(raw_dir / fname) as f:
        raw = json.load(f)
    for inc in raw:
        key = (service, inc.get("id", ""))
        raw_lookup[key] = inc

# count updates whose created_at is within 1h of incident start
WINDOW_SECONDS = 3600

first_hour_updates = []
for _, row in df.iterrows():
    key = (row["service"], row["id"])
    raw_inc = raw_lookup.get(key)
    if raw_inc is None:
        first_hour_updates.append(np.nan)
        continue

    created_str = raw_inc.get("created_at", "")
    updates = raw_inc.get("incident_updates", [])
    if not created_str:
        first_hour_updates.append(np.nan)
        continue

    try:
        t0 = datetime.fromisoformat(created_str.replace("Z", "+00:00"))
        count = 0
        for u in updates:
            ut_str = u.get("created_at", "")
            if ut_str:
                ut = datetime.fromisoformat(ut_str.replace("Z", "+00:00"))
                if (ut - t0).total_seconds() <= WINDOW_SECONDS:
                    count += 1
        first_hour_updates.append(count)
    except ValueError:
        first_hour_updates.append(np.nan)

df["first_hour_updates"] = first_hour_updates
matched = df["first_hour_updates"].notna().sum()
print(f"Matched first-hour updates for {matched}/{len(df)} incidents")


## 3. Dataset Overview

Quick look at the shape, types, and basic distributions before diving into specific analyses.

In [ ]:
print(f"Shape: {df.shape}")
print(f"Services: {df['service'].nunique()}")
print(f"Date range: {df['created_at'].min().date()} to {df['created_at'].max().date()}")
print()

print("=== Duration Stats ===")
print(df["duration_minutes"].describe().round(1))
print()

print("=== Impact Distribution ===")
print(df["impact"].value_counts())
print()

print("=== Service Distribution ===")
print(df["service"].value_counts())

## 4. Duration Distribution

Duration is the main variable I care about. Let's see how it's distributed -- the data is really skewed to the right, which I think is pretty normal for this kind of data.

### 4a. Histogram: Raw vs Log-Transformed

In [ ]:
durations = df["duration_minutes"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# I found this on stackoverflow
ax1.hist(durations, bins=20, color="#4ecdc4", alpha=0.8)
ax1.set_title("Duration Distribution (Raw)", fontsize=13, fontweight="bold")
ax1.set_xlabel("Duration (minutes)")
ax1.set_ylabel("Count")
ax1.axvline(durations.median(), color="#ff6b6b", linestyle="--", label=f"Median: {durations.median():.0f} min")
ax1.axvline(durations.mean(), color="#ffd93d", linestyle="--", label=f"Mean: {durations.mean():.0f} min")
ax1.legend(fontsize=9)

log_dur = np.log1p(durations)
ax2.hist(log_dur, bins=20, color="#a78bfa", edgecolor="white", alpha=0.85)
ax2.set_title("Duration Distribution (Log-Transformed)", fontsize=13, fontweight="bold")
ax2.set_xlabel("log(1 + duration)")
ax2.set_ylabel("Count")
ax2.axvline(log_dur.median(), color="#ff6b6b", linestyle="--", label=f"Median: {log_dur.median():.2f}")
ax2.legend(fontsize=9)

fig.suptitle("Incident Duration: the data is really skewed to the right", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "02a_duration_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

# skewness
skew_raw = stats.skew(durations, nan_policy="omit")
skew_log = stats.skew(log_dur, nan_policy="omit")
kurt_raw = stats.kurtosis(durations, nan_policy="omit")

print(f"Raw skewness:  {skew_raw:.2f}  (>2 = heavily right-skewed)")
print(f"Log skewness:  {skew_log:.2f}")
print(f"Raw kurtosis:  {kurt_raw:.2f}  (>3 = heavy tails)")
print(f"Mean/Median ratio: {durations.mean()/durations.median():.1f}x")

### 4b. Violin Plot by Impact Severity

In [ ]:
impact_order = ["none", "minor", "major", "critical"]
existing_impacts = [i for i in impact_order if i in df["impact"].values]

fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(
    data=df, x="impact", y="duration_minutes",
    order=existing_impacts, hue="impact", hue_order=existing_impacts,
    ax=ax, palette="magma", inner="box", linewidth=1.0, cut=0, legend=False
)
sns.stripplot(
    data=df, x="impact", y="duration_minutes",
    order=existing_impacts, ax=ax,
    color="white", size=4, alpha=0.6, jitter=True
)

ax.set_title("Duration by Impact Severity", fontsize=14, fontweight="bold")
ax.set_xlabel("Impact Level")
ax.set_ylabel("Duration (minutes)")

for i, imp in enumerate(existing_impacts):
    subset = df[df["impact"] == imp]["duration_minutes"]
    med = subset.median()
    ax.annotate(f"med={med:.0f}m\nn={len(subset)}", xy=(i, med),
                fontsize=8, ha="center", va="bottom", color="#ffd93d", fontweight="bold")

plt.tight_layout()
fig.savefig(FIG_DIR / "02c_duration_violin_by_impact.png", dpi=150)
plt.show()

### 4c. Cumulative Distribution Function (CDF)

In [ ]:
# this looks weird but it works
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(durations, bins=200, cumulative=True, density=True, histtype="step",
        color="#4ecdc4", linewidth=2.5)

# show percentage thresholds
thresholds = [30, 60, 120, 360, 1440]
for t in thresholds:
    pct = (durations <= t).mean() * 100
    if 0 < pct < 100:
        ax.axvline(t, color="#ff6b6b", linestyle=":", alpha=0.5)
        label = f"{t}min" if t < 1440 else f"{t//60}h"
        ax.annotate(f"{label}\n{pct:.0f}%", xy=(t, pct / 100), fontsize=9,
                    ha="left", va="bottom", color="#ffd93d", fontweight="bold",
                    xytext=(5, 5), textcoords="offset points")

ax.set_title("Cumulative Distribution: How Fast Do Incidents Resolve?", fontsize=14, fontweight="bold")
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Cumulative Proportion")
ax.set_xlim(left=0)
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "02d_duration_cdf.png", dpi=150, bbox_inches="tight")
plt.show()

pct_1h = (durations <= 60).mean() * 100
pct_6h = (durations <= 360).mean() * 100
print(f"{pct_1h:.0f}% resolve within 1 hour, {pct_6h:.0f}% within 6 hours")

## 5. Temporal Patterns

When do incidents happen? Is there a "don't deploy on Friday" effect? Let's check day-of-week, hour, and build a heatmap.

### 5a. Incidents by Day of Week

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["created_weekday"] = pd.Categorical(df["created_weekday"], categories=day_order, ordered=True)
day_counts = df["created_weekday"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(day_counts.index.astype(str), day_counts.values,
              color=sns.color_palette("Set2", 7), edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, day_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(day_counts) * 0.01,
            str(val), ha="center", va="bottom", fontsize=9, color="white")
ax.set_title("Incidents per Day of Week", fontsize=14, fontweight="bold")
ax.set_xlabel("Day")
ax.set_ylabel("Number of Incidents")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "temporal_day_of_week.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Peak day: {day_counts.idxmax()} ({day_counts.max()} incidents)")
print(f"Quietest: {day_counts.idxmin()} ({day_counts.min()} incidents)")

### 5b. Incidents by Hour (UTC)

In [ ]:
hour_counts = df["created_hour"].value_counts().sort_index().reindex(range(24), fill_value=0)

fig, ax = plt.subplots(figsize=(11, 5))
colors = ["#ff6b6b" if v == hour_counts.max() else "#4ecdc4" if v == hour_counts.min() else "#69b3a2"
          for v in hour_counts.values]
ax.bar(hour_counts.index, hour_counts.values, color=colors, edgecolor="white", linewidth=0.3)
ax.set_xticks(range(24))
ax.set_title("Incidents per Hour (UTC)", fontsize=14, fontweight="bold")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Number of Incidents")
ax.axhline(y=hour_counts.mean(), color="#ffd93d", linestyle="--", alpha=0.7,
           label=f"mean = {hour_counts.mean():.0f}")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "temporal_hour.png", dpi=150, bbox_inches="tight")
plt.show()

peak_hour = hour_counts.idxmax()
business = hour_counts.loc[9:17].sum()
offhours = hour_counts.sum() - business
print(f"Peak hour: {peak_hour}:00 UTC ({hour_counts.max()} incidents)")
print(f"Business hours (9-17): {business} ({business/hour_counts.sum()*100:.1f}%)")
print(f"Off-hours: {offhours} ({offhours/hour_counts.sum()*100:.1f}%)")

### 5c. Day x Hour Heatmap

In [ ]:
heatmap_data = df.groupby(["created_weekday", "created_hour"]).size().unstack(fill_value=0)
heatmap_data = heatmap_data.reindex(index=day_order, columns=range(24), fill_value=0)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(heatmap_data, cmap="YlOrRd", linewidths=0.3, linecolor="gray",
            annot=False, fmt="d", ax=ax, cbar_kws={"label": "Incident Count"})
ax.set_title("Incident Frequency: Day of Week x Hour (UTC)", fontsize=14, fontweight="bold")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("")
plt.tight_layout()
fig.savefig(FIG_DIR / "temporal_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

### 5d. Monthly Trend

In [ ]:
df["year_month"] = df["created_year"].astype(str) + "-" + df["created_month"].astype(str).str.zfill(2)
monthly = df.groupby("year_month").size().reset_index(name="count").sort_values("year_month")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly["year_month"], monthly["count"], marker="o", color="#4ecdc4", linewidth=2, markersize=6)
ax.fill_between(monthly["year_month"], monthly["count"], alpha=0.15, color="#4ecdc4")
ax.set_title("Monthly Incident Trend", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Number of Incidents")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "temporal_monthly_trend.png", dpi=150, bbox_inches="tight")
plt.show()

peak_month = monthly.loc[monthly["count"].idxmax()]
print(f"Peak month: {peak_month['year_month']} ({peak_month['count']} incidents)")
print(f"Total months covered: {len(monthly)}")

### 5e. Average Duration by Day of Week

Testing the classic "don't deploy on Friday" wisdom.

In [ ]:
df_valid = df[df["duration_minutes"].notna() & (df["duration_minutes"] > 0) & (~df["is_outlier"])].copy()
df_valid = df_valid.reset_index(drop=True)  # not sure if this is needed but just in case
day_duration = df_valid.groupby("created_weekday")["duration_minutes"].agg(["mean", "median", "count"])
day_duration = day_duration.reindex(day_order)

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(day_order))
width = 0.35
ax.bar([i - width/2 for i in x], day_duration["mean"], width, label="Mean", color="#ff6b6b",
       edgecolor="white", linewidth=0.5)
ax.bar([i + width/2 for i in x], day_duration["median"], width, label="Median", color="#4ecdc4",
       edgecolor="white", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(day_order, rotation=0)
ax.set_title("Avg Incident Duration by Day of Week (outliers removed)", fontsize=13, fontweight="bold")
ax.set_xlabel("Day")
ax.set_ylabel("Duration (minutes)")

overall_mean = df_valid["duration_minutes"].mean()
ax.axhline(y=overall_mean, color="#ffd93d", linestyle="--", alpha=0.7, label=f"overall mean = {overall_mean:.0f}")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "temporal_duration_by_day.png", dpi=150)
plt.show()

fri_mean = day_duration.loc["Friday", "mean"]
print(f"Friday mean: {fri_mean:.1f} min | Overall mean: {overall_mean:.1f} min")
print(f"Friday is close to average -- no strong Friday effect in this dataset")

## 6. Service Comparison

Which services have the worst (and best) incident response? Let's look at median durations and impact distributions.

### 6a. Median Duration by Service

In [ ]:
impact_colors = {"none": "#6bcb77", "minor": "#ffd93d", "major": "#ff9a3c", "critical": "#ff6b6b"}

services_sorted = df.groupby("service")["duration_minutes"].median().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = sns.color_palette("Set2", len(services_sorted))
bars = ax.barh(services_sorted.index, services_sorted.values,
               color=bar_colors, edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, services_sorted.values):
    ax.text(val + 2, bar.get_y() + bar.get_height() / 2, f"{val:.0f} min",
            va="center", fontsize=9, color="white")
ax.set_xlabel("Median Duration (minutes)", fontsize=11)
ax.set_title("Median Incident Duration by Service", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
fig.savefig(FIG_DIR / "04_median_duration_by_service.png", dpi=150, bbox_inches="tight")
plt.show()

### 6b. Impact Severity Distribution by Service (Stacked %)

In [ ]:
ct = pd.crosstab(df["service"], df["impact"])
for col in impact_order:
    if col not in ct.columns:
        ct[col] = 0
ct = ct[impact_order]
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct = ct_pct.loc[services_sorted.index]

fig, ax = plt.subplots(figsize=(10, 6))
bottom = np.zeros(len(ct_pct))
for impact in impact_order:
    vals = ct_pct[impact].values
    ax.barh(ct_pct.index, vals, left=bottom, label=impact.capitalize(),
            color=impact_colors[impact], edgecolor="white", linewidth=0.3)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 15:
            ax.text(b + v / 2, i, f"{v:.0f}%", ha="center", va="center",
                    fontsize=8, color="black", fontweight="bold")
    bottom += vals

ax.set_xlabel("Percentage of Incidents (%)", fontsize=11)
ax.set_title("Impact Severity Distribution by Service", fontsize=14, fontweight="bold", pad=15)
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim(0, 100)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_impact_distribution_by_service.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Impact & Class Imbalance

The `impact` column has 4 levels (none, minor, major, critical). I also create a binary target for ML: short (<60 min) vs long (>=60 min). Both have imbalance issues that matter for modeling.

In [ ]:
# binary target: short vs long
df["duration_class"] = np.where(df["duration_minutes"] < 60, "short", "long")
class_counts = df["duration_class"].value_counts()
class_pcts = (class_counts / len(df) * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ["#66c2a5", "#fc8d62"]
bars = ax.bar(class_counts.index, class_counts.values, color=bar_colors,
              edgecolor="white", linewidth=0.5, width=0.5)
for bar, cnt, pct in zip(bars, class_counts.values, class_pcts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"{cnt} ({pct}%)", ha="center", va="bottom", fontsize=12,
            fontweight="bold", color="white")
ax.set_xlabel("Duration Class", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Class Distribution: Short vs Long Incidents", fontsize=14, fontweight="bold")
ax.set_ylim(0, max(class_counts.values) * 1.25)
sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / "class_imbalance.png", dpi=150, bbox_inches="tight")
plt.show()

# imbalance analysis
majority = class_counts.idxmax()
minority = class_counts.idxmin()
ratio = class_counts[majority] / class_counts[minority]

print(f"Short (<60 min): {class_counts.get('short', 0)} ({class_pcts.get('short', 0)}%)")
print(f"Long  (>=60 min): {class_counts.get('long', 0)} ({class_pcts.get('long', 0)}%)")
print(f"Imbalance ratio: {ratio:.2f}:1")

if ratio < 1.5:
    print("Nearly balanced -- no special handling needed.")
elif ratio < 3:
    print("Mildly imbalanced -- stratified splits recommended.")
else:
    print("Significant imbalance -- model will bias toward majority. Need class weights or oversampling.")

## 8. Correlation Analysis

Let's see which numeric features correlate with duration. Using Spearman because the data is skewed.

### 8a. Spearman Correlation Heatmap

In [ ]:
numeric_cols = ["duration_minutes", "first_hour_updates", "num_components", "created_hour", "created_month"]
corr_spearman = df[numeric_cols].dropna().corr(method="spearman")

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_spearman, dtype=bool), k=1)
sns.heatmap(
    corr_spearman, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, vmin=-1, vmax=1,
    square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax,
)
ax.set_title("Spearman Correlation Heatmap (leakage-free features)", fontsize=14, pad=15)
plt.tight_layout()
fig.savefig(FIG_DIR / "corr_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

df_fh = df.dropna(subset=["first_hour_updates"])
rho, p = stats.spearmanr(df_fh["duration_minutes"], df_fh["first_hour_updates"])
print(f"Spearman rho between duration and first_hour_updates: {rho:.3f} (p={p:.4f})")


## 9. Hypothesis Testing

Three non-parametric tests since the data is heavily skewed. I'm using Mann-Whitney U for two-group comparisons and Kruskal-Wallis for the multi-group test. For effect size I'm just going to look at the medians since the difference looks small or large based on that.

### H3: Higher severity -> more status updates?

In [ ]:
print("H3 — Does first-hour update count differ across severity levels?")
print("  H0: first_hour_updates has the same distribution across impact groups")
print("  H1: at least one group differs\n")

severity_order = ["none", "minor", "major", "critical"]
df_h3 = df.dropna(subset=["first_hour_updates"]).copy()
df_h3["first_hour_updates"] = df_h3["first_hour_updates"].astype(int)

groups = {}
for level in severity_order:
    g = df_h3[df_h3["impact"] == level]["first_hour_updates"]
    if len(g) > 0:
        groups[level] = g
        print(f"  {level:>8}: n={len(g)}, median={g.median():.1f}, mean={g.mean():.2f}")

group_arrays = [groups[k] for k in groups]
k_groups = len(group_arrays)
n_total = sum(len(g) for g in group_arrays)

if k_groups >= 2:
    h_stat, p_val3 = kruskal(*group_arrays)
    # epsilon-squared effect size (Tomczak & Tomczak, 2014)
    eps_sq = (h_stat - k_groups + 1) / (n_total - k_groups) if n_total > k_groups else 0
    print(f"\nKruskal-Wallis H = {h_stat:.3f}, p = {p_val3:.4f}")
    print(f"Epsilon-squared effect size = {eps_sq:.3f}  (0.01=small, 0.08=medium, 0.26=large)")
    print(f"Decision at alpha=0.05: {'Reject H0' if p_val3 < 0.05 else 'Fail to reject H0'}")

    if p_val3 < 0.05:
        import scikit_posthocs as sp
        posthoc_df = pd.DataFrame({
            "value": pd.concat(group_arrays).values,
            "group": sum([[k] * len(v) for k, v in groups.items()], []),
        })
        dunn = sp.posthoc_dunn(posthoc_df, val_col="value", group_col="group", p_adjust="fdr_bh")
        print("\nDunn post-hoc (BH-corrected p-values):")
        print(dunn.round(4).to_string())
else:
    h_stat, p_val3, eps_sq = 0.0, 1.0, 0.0
    print("Not enough groups for Kruskal-Wallis test.")

fig, ax = plt.subplots(figsize=(8, 5))
test_colors = {"none": "#4a90e2", "minor": "#2ecc71", "major": "#f39c12", "critical": "#e74c3c"}
max_fhu = int(df_h3["first_hour_updates"].max()) if len(df_h3) else 1
bins3 = np.arange(-0.5, max_fhu + 1.5, 1)
for level in severity_order:
    if level in groups:
        ax.hist(groups[level], bins=bins3, alpha=0.6,
                label=f"{level} (n={len(groups[level])})", color=test_colors[level])
ax.set_xlabel("Updates in first hour")
ax.set_ylabel("Count")
ax.set_xticks(range(0, max_fhu + 1))
ax.set_title(f"H3: First-hour updates by severity (p={p_val3:.4f}, epsilon-sq={eps_sq:.3f})")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "h3_severity_updates.png", dpi=150)
plt.show()


## 10. Data Leakage Analysis

Here's a subtle but important problem: `num_updates` counts ALL status updates, including ones posted AFTER the incident resolved. If we use this to predict duration, we're leaking future information -- this is kind of cheating because longer incidents naturally get more updates, so the model just learns "more updates = longer".

**Fix:** Only count updates from the first hour of the incident. This simulates what we'd actually know early on.

In [ ]:
# compare correlations: leaky num_updates vs clean first_hour_updates
df_leak = df.dropna(subset=["first_hour_updates", "duration_minutes"]).copy()
df_leak["first_hour_updates"] = df_leak["first_hour_updates"].astype(int)

rho_leaky, p_leaky = stats.spearmanr(df_leak["num_updates"], df_leak["duration_minutes"])
rho_clean, p_clean = stats.spearmanr(df_leak["first_hour_updates"], df_leak["duration_minutes"])

print("Spearman correlation with duration_minutes (consistent with rest of notebook):")
print(f"  num_updates (LEAKY):        rho = {rho_leaky:.4f}  (p={p_leaky:.4g})")
print(f"  first_hour_updates (CLEAN): rho = {rho_clean:.4f}  (p={p_clean:.4g})")

diff_mask = df_leak["num_updates"] != df_leak["first_hour_updates"]
print(f"\nIncidents where leaky != clean: {diff_mask.sum()}/{len(df_leak)}")
print("These are cases where more updates came AFTER the first hour")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_leak["num_updates"], df_leak["duration_minutes"],
                c="#e74c3c", alpha=0.55, s=50)
axes[0].set_yscale("log")
axes[0].set_xlabel("num_updates (ALL updates — LEAKY)", fontsize=11)
axes[0].set_ylabel("Duration (minutes, log scale)", fontsize=11)
axes[0].set_title(f"LEAKY feature\nSpearman rho = {rho_leaky:.3f}", fontsize=12, color="#e74c3c")

axes[1].scatter(df_leak["first_hour_updates"], df_leak["duration_minutes"],
                c="#27ae60", alpha=0.55, s=50)
axes[1].set_yscale("log")
axes[1].set_xlabel("first_hour_updates (first 1h only — CLEAN)", fontsize=11)
axes[1].set_ylabel("Duration (minutes, log scale)", fontsize=11)
axes[1].set_title(f"CLEAN feature\nSpearman rho = {rho_clean:.3f}", fontsize=12, color="#27ae60")

fig.suptitle("Data leakage: total updates vs time-constrained updates", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "leakage_scatter_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## 11. Component Analysis

Which components are most commonly affected during incidents?

In [ ]:
# flatten components lists
all_components = []
for comps in df["components"]:
    if isinstance(comps, list):
        all_components.extend(comps)
    elif isinstance(comps, str) and comps.startswith("["):
        try:
            parsed = json.loads(comps.replace("'", '"'))
            all_components.extend(parsed)
        except Exception:
            pass

comp_counts = pd.Series(all_components).value_counts().head(20)

if len(comp_counts) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    comp_counts.plot(kind="barh", ax=ax, color=sns.color_palette("Set2", len(comp_counts)))
    ax.set_xlabel("Number of Incidents")
    ax.set_title("Top 20 Most Affected Components", fontsize=14, fontweight="bold")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "07a_top_components.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Total unique components: {pd.Series(all_components).nunique()}")
else:
    print("No component data found")

In [ ]:
# save final enriched dataset
df.to_csv(DATA_DIR / "incidents_clean.csv", index=False)
print(f"Saved incidents_clean.csv: {len(df)} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")


## 12. Key Findings

- **Median duration: 82 min, mean: 480 min** -- a few mega-outages pull the mean way above the median. The data is really skewed to the right, log-transform helps.
- **39% of incidents resolve within 1 hour** -- the CDF shows most are done within a few hours, but the tail is really long.
- **Spearman rho=0.462 between duration and num_updates** -- but this is mostly data leakage (this is kind of cheating because longer incidents get more updates by definition). After fixing with the first-hour window, the correlation drops a lot.
- **Impact severity matters** -- critical/major incidents tend to last longer. Kruskal-Wallis test was highly significant (p < 0.0001).
- **No strong "Friday deploy" effect** -- despite the meme, Friday incidents aren't significantly longer in this dataset.
- **Class imbalance exists** -- the short/long binary target is imbalanced, which will need stratified splits and possibly class weights in the ML phase.
- **We might be missing real effects because the sample is small** -- with ~700 incidents after cleaning, statistical power is limited.

## 13. Limitations

- **Small sample size** (~700 incidents after filtering). This limits the power of hypothesis tests -- we might be missing real effects because the sample is small.
- **Observational data** -- correlations don't imply causation. A service having longer outages might be because it's more complex, not because it's poorly managed.
- **Status page data is self-reported** -- companies control what they post. Some might underreport or delay acknowledgment.
- **Temporal features are cyclical** -- created_hour wraps around (23 -> 0), so linear correlation isn't ideal for it. I noted this but didn't apply circular statistics.
- **Impact labels aren't standardized** -- each company defines "critical" vs "major" differently. Cross-service comparisons should be taken with a grain of salt.
- **No external context** -- I don't know if an outage coincided with a major release, DDoS attack, or infra migration. That context would help explain duration differences.

## 14. Next Steps (ML Plan)

1. **Feature engineering** -- use `first_hour_updates` instead of `num_updates` to avoid leakage. Add time-based features (is_business_hours, is_weekend).
2. **Binary classification** -- predict short vs long incidents. Try logistic regression as baseline, then random forest / gradient boosting.
3. **Handle imbalance** -- stratified K-fold, class weights, maybe SMOTE if needed.
4. **Evaluation** -- focus on precision/recall/F1 rather than accuracy (because of class imbalance). Build a confusion matrix and look at which incidents the model gets wrong.
5. **Feature importance** -- use SHAP or permutation importance to understand what drives predictions.
6. **Cross-validation** -- small dataset means I can't afford a huge test set. Use 5-fold stratified CV.

---

*End of EDA. All figures saved to `figures/`.*